<a href="https://colab.research.google.com/github/UgamThakkar/banking-intent-classifier/blob/main/03_error_analysis_confusion_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install transformers datasets -q


In [6]:
import torch
print("Gpu Available", torch.cuda.is_available())
print("GPU name: ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Gpu Available True
GPU name:  Tesla T4


In [7]:
import pandas as pd
train_df = pd.read_csv("https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv")
test_df = pd.read_csv("https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv")

print("Training shape: ", train_df.shape)
print("Testing shape: ", test_df.shape)
train_df.head()

Training shape:  (10003, 2)
Testing shape:  (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [8]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder() #scans all 77 categories and assigns them a unique integer from 0 to 76
label_encoder.fit(train_df['category']) #fit learns the mapping of words -> numbers
train_df['label'] = label_encoder.transform(train_df['category']) #.transform actually applies this transformation of words to numbers
test_df['label'] = label_encoder.transform(test_df['category'])

print("Number of unique intents ", len(label_encoder.classes_))
print()
print("example mapping: ")
for i in range(3):
  print(f" '{train_df['category'].iloc[i]}' -> {train_df['label'].iloc[i]}")


Number of unique intents  77

example mapping: 
 'card_arrival' -> 12
 'card_arrival' -> 12
 'card_arrival' -> 12


In [9]:
import json

label_map = {int(i): label for i, label in enumerate(label_encoder.classes_)}
print(label_map[0], label_map[1], label_map[2])

Refund_not_showing_up activate_my_card age_limit


In [10]:
from transformers import AutoTokenizer #autotokenizer is a smart loader we provide a model name to it and it figures out which exact tokenizer the model needs and fetches it

MODEL_NAME = "distilbert-base-uncased" #pretrained model using for this project
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [11]:
example_text = train_df['text'].iloc[0]

encoded = tokenizer(
    example_text,
    padding='max_length',
    truncation=True,
    max_length=32,
    return_tensors='pt',
)

print("Original text:", example_text)
print()
print("input_ids:")
print(encoded['input_ids'])
print()
print("attention_mask:")
print(encoded['attention_mask'])

Original text: I am still waiting on my card?

input_ids:
tensor([[ 101, 1045, 2572, 2145, 3403, 2006, 2026, 4003, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])


In [12]:
decoded = tokenizer.decode(encoded['input_ids'][0])
print(decoded)

[CLS] i am still waiting on my card? [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [13]:
def tokenize_batch(texts):
  return tokenizer(
      texts.tolist(),
      padding = 'max_length',
      truncation = True,
      max_length = 32,
      return_tensors = 'pt',
  )

train_encodings = tokenize_batch(train_df['text'])
test_encodings = tokenize_batch(test_df['text'])
print("Train input_ids Shape: ", train_encodings['input_ids'].shape)
print("Test input_ids Shape: ", test_encodings['input_ids'].shape)

Train input_ids Shape:  torch.Size([10003, 32])
Test input_ids Shape:  torch.Size([3080, 32])


In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 77,
)
print(model.config)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17",
    "18": "LABEL_18",
    "19": "LABEL_19",
    "20": "LABEL_20",
    "21": "LABEL_21",
    "22": "LABEL_22",
    "23": "LABEL_23",
    "24": "LABEL_24",
    "25": "LABEL_25",
    "26": "LABEL_26",
    "27": "LABEL_27",
    "28": "LABEL_28",
    "29": "LABEL_29",
    "30": "LABEL_30",
    "31": "LABEL_31",
    "32": "LABEL_32",
    "33": "LABEL_33",
    "34

In [16]:
from torch.utils.data import DataLoader
import torch
class TicketDataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['labels'] = self.labels[idx]
    return item

train_dataset = TicketDataset(train_encodings, torch.tensor(train_df['label'].values))
test_dataset = TicketDataset(test_encodings, torch.tensor(test_df['label'].values))
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Number of training examples:", len(train_dataset))
print()
print("A single example looks like:")
example = train_dataset[0]
for key, value in example.items():
    print(f"  {key}: shape={value.shape}, dtype={value.dtype}")

Number of training examples: 10003

A single example looks like:
  input_ids: shape=torch.Size([32]), dtype=torch.int64
  token_type_ids: shape=torch.Size([32]), dtype=torch.int64
  attention_mask: shape=torch.Size([32]), dtype=torch.int64
  labels: shape=torch.Size([]), dtype=torch.int64


In [17]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

batch = next(iter(train_loader))
for key,value in batch.items():
  print(f"{key}: shape = {value.shape}")

input_ids: shape = torch.Size([16, 32])
token_type_ids: shape = torch.Size([16, 32])
attention_mask: shape = torch.Size([16, 32])
labels: shape = torch.Size([16])


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model is on:", next(model.parameters()).device)

Model is on: cuda:0


In [19]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr = 5e-5)

In [20]:
model.train()

EPOCHS = 3

for epoch in range(EPOCHS):
  total_loss = 0

  for batch in train_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()
    outputs = model(
        input_ids = input_ids,
        attention_mask = attention_mask,
        labels = labels,
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    total_loss+=loss.item()

  avg_loss = total_loss/len(train_loader)
  print(f"Epoch {epoch+1}/EPOCHS - average loss: {avg_loss: .4f}")

Epoch 1/EPOCHS - average loss:  1.8527
Epoch 2/EPOCHS - average loss:  0.3639
Epoch 3/EPOCHS - average loss:  0.1813


In [21]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds= []
all_labels = []

with torch.no_grad():
  for batch in test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids = input_ids, attention_mask = attention_mask)
    predictions = torch.argmax(outputs.logits, dim=1)
    all_preds.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy: .4f}")

Test Accuracy:  0.9006


In [22]:
model.save_pretrained('banking_intent_model')
tokenizer.save_pretrained('banking_intent_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('banking_intent_model/tokenizer_config.json',
 'banking_intent_model/tokenizer.json')

In [23]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(all_labels, all_preds)

label_names = [label_map[i] for i in range(77)]
confusions = []

for true_idx in range(77):
  row = cm[true_idx].copy()
  if row.sum() > 0:
    predicted_idx = row.argmax()
    confusions.append((
        label_names[true_idx],
        label_names[predicted_idx],
        row[predicted_idx],
        ))
confusions.sort(key=lambda x: -x[2])

print("Top 10 most common confusions (true label -> predicted label, count):")
for true_label, pred_label, count in confusions[:10]:
    print(f"  {true_label:30s} -> {pred_label:30s}  ({count} times)")

Top 10 most common confusions (true label -> predicted label, count):
  age_limit                      -> age_limit                       (40 times)
  apple_pay_or_google_pay        -> apple_pay_or_google_pay         (40 times)
  country_support                -> country_support                 (40 times)
  edit_personal_details          -> edit_personal_details           (40 times)
  get_physical_card              -> get_physical_card               (40 times)
  passcode_forgotten             -> passcode_forgotten              (40 times)
  terminate_account              -> terminate_account               (40 times)
  transaction_charged_twice      -> transaction_charged_twice       (40 times)
  verify_source_of_funds         -> verify_source_of_funds          (40 times)
  verify_top_up                  -> verify_top_up                   (40 times)


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=5e-5)
print("Model on:", next(model.parameters()).device)

Model on: cuda:0


In [25]:
model.train()
EPOCHS = 3

for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} — average loss: {total_loss/len(train_loader):.4f}")

Epoch 1/3 — average loss: 0.1140
Epoch 2/3 — average loss: 0.0765
Epoch 3/3 — average loss: 0.0634


In [26]:
from sklearn.metrics import accuracy_score

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Test accuracy:", accuracy_score(all_labels, all_preds))

Test accuracy: 0.913961038961039


In [27]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(all_labels, all_preds)

label_names = [label_map[i] for i in range(77)]

confusions = []
for true_idx in range(77):
    row = cm[true_idx].copy()
    row[true_idx] = 0  # zero out the correct answer, we only want mistakes
    if row.sum() > 0:
        predicted_idx = row.argmax()
        confusions.append((
            label_names[true_idx],
            label_names[predicted_idx],
            row[predicted_idx],
        ))

confusions.sort(key=lambda x: -x[2])

print("Top 10 most common confusions (true label -> predicted label, count):")
for true_label, pred_label, count in confusions[:10]:
    print(f"  {true_label:30s} -> {pred_label:30s}  ({count} times)")

Top 10 most common confusions (true label -> predicted label, count):
  why_verify_identity            -> verify_my_identity              (7 times)
  card_not_working               -> activate_my_card                (6 times)
  lost_or_stolen_card            -> card_arrival                    (6 times)
  extra_charge_on_statement      -> card_payment_fee_charged        (5 times)
  fiat_currency_support          -> exchange_via_app                (5 times)
  balance_not_updated_after_bank_transfer -> pending_transfer                (4 times)
  card_acceptance                -> country_support                 (4 times)
  card_arrival                   -> card_delivery_estimate          (4 times)
  compromised_card               -> lost_or_stolen_card             (4 times)
  declined_cash_withdrawal       -> wrong_amount_of_cash_received   (4 times)


In [28]:
# Get the integer label IDs for these two intents
card_not_working_id = label_encoder.transform(['card_not_working'])[0]
activate_my_card_id = label_encoder.transform(['activate_my_card'])[0]

# Find test examples where the TRUE label was card_not_working
# but the model PREDICTED activate_my_card
all_preds_arr = np.array(all_preds)
all_labels_arr = np.array(all_labels)

confused_mask = (all_labels_arr == card_not_working_id) & (all_preds_arr == activate_my_card_id)
confused_indices = np.where(confused_mask)[0]

print(f"Found {len(confused_indices)} confused examples\n")

for idx in confused_indices:
    text = test_df['text'].iloc[idx]
    print(f"Text: {text}")
    print(f"True label: card_not_working  |  Predicted: activate_my_card")
    print()

Found 6 confused examples

Text: I can't get my card to work.
True label: card_not_working  |  Predicted: activate_my_card

Text: How can I get my physical card to work?
True label: card_not_working  |  Predicted: activate_my_card

Text: Can a card stop working?
True label: card_not_working  |  Predicted: activate_my_card

Text: My card stopped working
True label: card_not_working  |  Predicted: activate_my_card

Text: My card suddenly quit working.
True label: card_not_working  |  Predicted: activate_my_card

Text: How can i check if my card is working?
True label: card_not_working  |  Predicted: activate_my_card

